In [1]:
# Cell 1: Setup
import sys, os, time, math, random, subprocess
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root("src")

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation
Device: cuda


In [ ]:
# Configuration 
 
# ---- datasets ----
DATASETS = {
    "DRIVE":     dict(root="../../data/raw/DRIVE",     label_folder="1st_manual"),
    "CHASE-DB1": dict(root="../../data/raw/CHASE_DB1", label_folder="1st_manual"),
    "STARE":     dict(root="../../data/raw/STARE",     label_folder="2nd_manual"),
}

# ---- checkpoints ----
CKPTS = {
    "DRIVE": {
        "unet":     "./outputs/checkpoints/unet/[DRIVE_UNET] base_unet.pth",
        "proposed": "./outputs/checkpoints/[DRIVE] MATHFI.pth",
    },
    "CHASE-DB1": {
        "unet":     "./outputs/checkpoints/unet/[CHASEDB1_UNET] base_unet.pth",
        "proposed": "./outputs/checkpoints/[CHASEDB1] MATHFI.pth",
    },
    "STARE": {
        "unet":     "./outputs/checkpoints/unet/[STARE_UNET] base_unet.pth",
        "proposed": "./outputs/checkpoints/[STARE] MATHFI.pth",
    },
}

# ---- evaluation hyperparams ----
IMAGE_SIZE = 512      # val/test preprocessing size
TAU        = 0.5      # decision threshold for binarizing probs
NUM_WORKERS = 0       # safe locally; bump on Linux if desired
BATCH_SIZE  = 1
WINDOW      = IMAGE_SIZE   # sliding-window tile size
OVERLAP     = 0.25

# ---- proposed model (DPCN) hyperparams per dataset (MUST MATCH TRAINING) ----
# If trained the same way for all datasets, keep a single block.
PROPOSED_CFG = {
    "default": dict(enh_channels=64, iters=6, threshold_mode="scaled_vat", half_life=2.0, reduce_to=64),
    # Example: override per dataset (uncomment and edit if needed)
    # "DRIVE":   dict(enh_channels=32, iters=8, threshold_mode="scaled_vat", half_life=4.2, reduce_to=56),
    # "STARE":   dict(enh_channels=64, iters=6, threshold_mode="scaled_vat", half_life=2.8, reduce_to=64),
}

# ---- base UNet extras (as used in your code) ----
BASE_KW = {"cbam_reduction": 16}

# Where to dump CSVs (optional)
OUT_DIR = Path("./eval_exports"); OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Project imports & model builders

from src.data.prepare_dataset import build_pairs_for_split
from src.data.dataloader import make_loaders
from src.data.augmentations import get_val_augs

# Baseline: adjust import to your baseline class if different
from src.models.unet import UNet as BaselineUNet

# Proposed wrapper
from src.models.wrappers.dpcn_concat_unet import DPCNConcatUNet

def build_model(model_name: str, dataset_name: str):
    if model_name == "unet":
        m = BaselineUNet(in_channels=1)
    elif model_name == "proposed":
        cfg = PROPOSED_CFG.get(dataset_name, PROPOSED_CFG["default"])
        m = DPCNConcatUNet(
            in_ch=1,
            enh_channels=cfg["enh_channels"],
            iters=cfg["iters"],
            threshold_mode=cfg["threshold_mode"],
            half_life=cfg["half_life"],
            reduce_to=cfg["reduce_to"],
            base_kwargs=BASE_KW
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m.to(DEVICE).eval()

def load_weights(model: torch.nn.Module, path: str):
    if not Path(path).exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    return model.eval()
